**Student Performance Dashboard**

In [ ]:
# Tasks in the project:
# Phase 1 — Data Exploration
#  - basic stats
#  - missing values
#  - unique values

# Phase 2 — Filtering & Analysis
#  - top performers
#  - failing students
#  - subject-wise averages
#  - department-wise analysis

# Phase 3 — Feature Engineering
#  - total marks
#  - percentage
#  - grade column

# Phase 4 — Ranking
#  - class rank
#  - department rank

# Phase 5 — Insights
#  - best department
#  - hardest subject
#  - distribution

In [ ]:
# creating a realistic synthetic student dataset
import pandas as pd
import numpy as np

# creating a random number generator instance
rng = np.random.default_rng(seed=12345)

# creating a dataset with columns : name, marks of multiple subjects, attendance and department (data may be chosen randomly through numpy's random module)
dataset = pd.DataFrame({"Name":["Sam","Henry","Ray","Aaron","Ryan"], "Python fundamentals":rng.integers(0,101,5), "System Design":rng.integers(0,101,5), "Machine Learning":rng.integers(0,101,5), "Blockchain technology":rng.integers(0,101,5), "Attendance":rng.integers(0,101,5), "Department":np.random.choice(["CSE","AIDS","AIML","ECE"],5)})
dataset.index = dataset["Name"]
dataset.index.name = "Students"
del dataset["Name"]

# putting some nans in the data at random positions
random_idx_row = rng.integers(0, len(dataset.index),3)
random_idx_col = rng.integers(1, len(dataset.columns)-1,3)
random_idx = zip(random_idx_row, random_idx_col)
random_idx = list(random_idx)

for i,j in random_idx:
  dataset.iloc[i,j] = np.nan

print(dataset.head())

          Python fundamentals  System Design  Machine Learning  \
Students                                                         
Sam                        70            NaN               NaN   
Henry                      22           64.0              33.0   
Ray                        79           68.0              57.0   
Aaron                      31           99.0               NaN   
Ryan                       20           39.0              21.0   

          Blockchain technology  Attendance Department  
Students                                                
Sam                          18          71       AIML  
Henry                        23          25        CSE  
Ray                          67          92       AIML  
Aaron                        61          95        ECE  
Ryan                         95          73        ECE  


Phase 1 - Data Exploration

In [ ]:
# basic stats
dataset.iloc[:,:-2].describe() # passing only numeric columns of dataset since .describe() method only supports the same

,Python fundamentals,System Design,Machine Learning,Blockchain technology
count,5.000000,4.000000,3.000000,5.00000
mean,44.400000,67.500000,37.000000,52.80000
std,27.969626,24.610296,18.330303,32.20559
min,20.000000,39.000000,21.000000,18.00000
25%,22.000000,57.750000,27.000000,23.00000
50%,31.000000,66.000000,33.000000,61.00000
75%,70.000000,75.750000,45.000000,67.00000
max,79.000000,99.000000,57.000000,95.00000


In [ ]:
# missing values
# printing count of missing values column wise
print("null values (column wise):")
print(dataset.isnull().sum(), "\n")

# printing count of missing values row wise
print("null values (row wise):")
print(dataset.isnull().sum(axis=1), "\n")

# printing total number of missing values in the dataset
print("total null values:", dataset.isnull().sum().sum(), "\n")
# print(dataset.isnull().sum(axis=1).sum()) # returns the same value as above line

# filling missing values
# imputing nan values with mean score of student other than subject with nan value
dataset_imputed = dataset.iloc[:,:-2].T.copy()
# dataset_imputed.fillna(pd.Series(np.nanmean(dataset.iloc[:,:-1], axis=1)), axis=1, inplace=True) # won't work since fillna method matches input series index with the columns of the dataframe
dataset_imputed.fillna(pd.Series(dataset.iloc[:,:-2].mean(axis=1)), inplace=True) # fills null values with mean (calculated without considering nan values) student-wise, used .mean instead of np.nanmean since it preserves the index labels properly while np.nanmean() may create a new integer index (0,1,2,3,4,....)
dataset_imputed = dataset_imputed.T
print("student wise mean marks scored across non-null subjects:")
print(dataset.iloc[:,:-1].mean(axis=1))

dataset.iloc[:,:-2] = dataset_imputed # filling the imputed df back in the original dataframe
print(dataset)

null values (column wise):
Python fundamentals      0
System Design            1
Machine Learning         2
Blockchain technology    0
Attendance               0
Department               0
dtype: int64 

null values (row wise):
Students
Sam      2
Henry    0
Ray      0
Aaron    1
Ryan     0
dtype: int64 

total null values: 3 

student wise mean marks scored across non-null subjects:
Students
Sam      53.0
Henry    33.4
Ray      72.6
Aaron    71.5
Ryan     49.6
dtype: float64
          Python fundamentals  System Design  Machine Learning  \
Students                                                         
Sam                      70.0           44.0         44.000000   
Henry                    22.0           64.0         33.000000   
Ray                      79.0           68.0         57.000000   
Aaron                    31.0           99.0         63.666667   
Ryan                     20.0           39.0         21.000000   

          Blockchain technology  Attendance Department  

/tmp/ipykernel_1019/390838050.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'Students
Sam      70.0
Henry    22.0
Ray      79.0
Aaron    31.0
Ryan     20.0
Name: Python fundamentals, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataset.iloc[:,:-2] = dataset_imputed # filling the imputed df back in the original dataframe
/tmp/ipykernel_1019/390838050.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'Students
Sam      18.0
Henry    23.0
Ray      67.0
Aaron    61.0
Ryan     95.0
Name: Blockchain technology, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dataset.iloc[:,:-2] = dataset_imputed # filling the imputed df back in the original dataframe


In [ ]:
# unique values
print("unique departments:", dataset["Department"].unique())
print("total number of unique departments in the dataset:", len(dataset["Department"].unique()))

unique departments: ['AIML' 'CSE' 'ECE']
total number of unique departments in the dataset: 3


Phase 2 - Filtering and analysis

In [ ]:
# top performers
# ranking students according to their percentage
dataset["%"] = ((dataset["Python fundamentals"] + dataset["System Design"] + dataset["Machine Learning"] + dataset["Blockchain technology"])/4).round(4)
dataset["rank"] = dataset["%"].rank(ascending=False, method="dense") # student with highest % gets rank 1 and in case of equal %, both are assigned equal rank and next rank goes with the next number
dataset.sort_values("rank", inplace=True)
print(dataset.head())

          Python fundamentals  System Design  Machine Learning  \
Students                                                         
Ray                      79.0           68.0         57.000000   
Aaron                    31.0           99.0         63.666667   
Sam                      70.0           44.0         44.000000   
Ryan                     20.0           39.0         21.000000   
Henry                    22.0           64.0         33.000000   

          Blockchain technology  Attendance Department        %  rank  
Students                                                               
Ray                        67.0          92       AIML  67.7500   1.0  
Aaron                      61.0          95        ECE  63.6667   2.0  
Sam                        18.0          71       AIML  44.0000   3.0  
Ryan                       95.0          73        ECE  43.7500   4.0  
Henry                      23.0          25        CSE  35.5000   5.0  


In [ ]:
# failing students
dataset[dataset.loc[:,"%"]<40]

,Python fundamentals,System Design,Machine Learning,Blockchain technology,Attendance,Department,%,rank
Students,,,,,,,,
Henry,22.0,64.0,33.0,23.0,25,CSE,35.5,5.0


In [ ]:
# student wise averages
dataset.iloc[:,:-3] = dataset.iloc[:,:-3].round(4) # rounding decimal values to 4 digits
dataset.iloc[:,:-3].mean(axis=1)

,0
Students,
Ray,72.60000
Aaron,69.93334
Sam,49.40000
Ryan,49.60000
Henry,33.40000


In [ ]:
# department wise analysis
# department wise average total marks
dataset
dataset["total_marks"] = dataset["Python fundamentals"] + dataset["System Design"] + dataset["Machine Learning"] + dataset["Blockchain technology"]
dept_list = dataset["Department"].unique()
avg_marks_dept_wise = [(dataset.loc[dataset["Department"]==dept,"total_marks"]).mean() for dept in dept_list]
# dept_dict = {key:value for key,value in zip(dept_list, avg_marks_dept_wise)}
dept_dict = dict(zip(dept_list, avg_marks_dept_wise))
print("department wise average total marks:")
print(pd.Series(dept_dict), "\n")
print("department wise average %:")
print(list((dept, pct) for dept, pct in zip(dept_list,list((dataset.loc[dataset["Department"]==dept,"%"].mean().round(2) for dept in dataset["Department"].unique())))))
dataset

# dataset.loc[dataset["Department"]=="CSE"]
# dataset.loc[dataset["Department"]=="CSE", "total_marks"]
# dataset.loc[dataset["Department"]=="CSE", "total_marks"].mean()

department wise average total marks:
AIML    223.50000
ECE     214.83335
CSE     142.00000
dtype: float64 

department wise average %:
[('AIML', np.float64(55.88)), ('ECE', np.float64(53.71)), ('CSE', np.float64(35.5))]


,Python fundamentals,System Design,Machine Learning,Blockchain technology,Attendance,Department,%,rank,total_marks
Students,,,,,,,,,
Ray,79.0,68.0,57.0000,67.0,92,AIML,67.7500,1.0,271.0000
Aaron,31.0,99.0,63.6667,61.0,95,ECE,63.6667,2.0,254.6667
Sam,70.0,44.0,44.0000,18.0,71,AIML,44.0000,3.0,176.0000
Ryan,20.0,39.0,21.0000,95.0,73,ECE,43.7500,4.0,175.0000
Henry,22.0,64.0,33.0000,23.0,25,CSE,35.5000,5.0,142.0000


Feature Engineering

In [ ]:
# total_marks column already created
# % column already created

# calculating grades of students
# criteria for grades:
# - 91-100% : 'A'
# - 81-90 % : 'B'
# - 61-80 % : 'C'
# - 41-60 % : 'D'
# - 0-40 % : 'E'

grades_list = ["A","B","C","D","E"]
conditions_list = [(dataset["%"]>=91), (dataset["%"]>=91) & (dataset["%"]<=90), (dataset["%"]>=61) & (dataset["%"]<=80), (dataset["%"]>=41) & (dataset["%"]<=60), (dataset["%"]>=0) & (dataset["%"]<=40)]
# grades_dict = {grade: condition for grade, condition in zip(grades_list, conditions_list)}

dataset["grade"] = np.select(conditions_list, grades_list, default="Unknown") # assigns grades row wise according to the student's % values
dataset


,Python fundamentals,System Design,Machine Learning,Blockchain technology,Attendance,Department,%,rank,total_marks,grade
Students,,,,,,,,,,
Ray,79.0,68.0,57.0000,67.0,92,AIML,67.7500,1.0,271.0000,C
Aaron,31.0,99.0,63.6667,61.0,95,ECE,63.6667,2.0,254.6667,C
Sam,70.0,44.0,44.0000,18.0,71,AIML,44.0000,3.0,176.0000,D
Ryan,20.0,39.0,21.0000,95.0,73,ECE,43.7500,4.0,175.0000,D
Henry,22.0,64.0,33.0000,23.0,25,CSE,35.5000,5.0,142.0000,E


Ranking

In [ ]:
# class rank is already calculated
# calculating department rank

# dept_wise_students = {dept:students for dept,students in zip(dept_list,list((dataset.index[dataset["Department"]==dept_], dataset.loc[dataset.index[dataset["Department"]==dept_],"%"]) for dept_ in dept_list))} # creates a dictionary with department names as keys and values as list of index objects (with values student names)
dept_wise_student_ranks = {dept:students for dept,students in zip(dept_list,list((dataset.index[dataset["Department"]==dept_], dataset.loc[dataset.index[dataset["Department"]==dept_],"%"].rank(ascending=False, method="dense")) for dept_ in dept_list))}
print(dept_wise_student_ranks)
# print(dept_wise_student_ranks["CSE"][1][0])
print(dept_wise_student_ranks["AIML"]) # we have a dictionary now with keys as dept name and values as tuple of index objects and a series (index objects and series also have corresponding values)
print(dept_wise_student_ranks["AIML"][0])
print(dept_wise_student_ranks["AIML"][1])
print(dept_wise_student_ranks["AIML"][0][0])
print(dept_wise_student_ranks["AIML"][1][1])
dataset
# dataset["dept_rank"] = dataset["Department"].map(dept_wise_student_ranks)
# creating a custom function for nested lookup of dept_rank values of students
def nested_lookup(row):
  if row["Department"] in dept_wise_student_ranks:
    # for student in dept_wise_student_ranks[row["Department"]][1]:
    #   if student==row.name:
        return dept_wise_student_ranks[row["Department"]][1][row.name]

dataset["dept_rank"] = dataset.apply(nested_lookup, axis=1)
dataset



{'AIML': (Index(['Ray', 'Sam'], dtype='object', name='Students'), Students
Ray    1.0
Sam    2.0
Name: %, dtype: float64), 'ECE': (Index(['Aaron', 'Ryan'], dtype='object', name='Students'), Students
Aaron    1.0
Ryan     2.0
Name: %, dtype: float64), 'CSE': (Index(['Henry'], dtype='object', name='Students'), Students
Henry    1.0
Name: %, dtype: float64)}
(Index(['Ray', 'Sam'], dtype='object', name='Students'), Students
Ray    1.0
Sam    2.0
Name: %, dtype: float64)
Index(['Ray', 'Sam'], dtype='object', name='Students')
Students
Ray    1.0
Sam    2.0
Name: %, dtype: float64
Ray
2.0


/tmp/ipykernel_1019/6507786.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(dept_wise_student_ranks["AIML"][1][1])


,Python fundamentals,System Design,Machine Learning,Blockchain technology,Attendance,Department,%,rank,total_marks,grade,dept_rank
Students,,,,,,,,,,,
Ray,79.0,68.0,57.0000,67.0,92,AIML,67.7500,1.0,271.0000,C,1.0
Aaron,31.0,99.0,63.6667,61.0,95,ECE,63.6667,2.0,254.6667,C,1.0
Sam,70.0,44.0,44.0000,18.0,71,AIML,44.0000,3.0,176.0000,D,2.0
Ryan,20.0,39.0,21.0000,95.0,73,ECE,43.7500,4.0,175.0000,D,2.0
Henry,22.0,64.0,33.0000,23.0,25,CSE,35.5000,5.0,142.0000,E,1.0


Phase 5 - Insights

In [ ]:
# best department
# considering highest total marks per department as the criteria

dept_total_marks_dict = {dept: total_marks for dept, total_marks in zip(dept_list, list(dataset.loc[dataset["Department"]==dept_,"total_marks"].sum() for dept_ in dept_list))}
print(dept_total_marks_dict)
dept_total_marks_dict = pd.Series(dept_total_marks_dict)
print(dept_total_marks_dict, "\n")
dept_ranks = dept_total_marks_dict.rank(ascending=False, method="dense")
print("department ranks according to total marks:")
print(dept_ranks, "\n")

# considering highest average % per department as the criteria
dept_wise_average_pct = {dept: pct for dept, pct in zip(dept_list, list(dataset.loc[dataset["Department"]==dept_, "%"].mean() for dept_ in dept_list))}
print(dept_wise_average_pct)
dept_wise_average_pct = pd.Series(dept_wise_average_pct)
print(dept_wise_average_pct, "\n")
dept_ranks_pct = dept_wise_average_pct.rank(ascending=False, method="dense")
print("department ranks according to average %:")
print(dept_ranks_pct)

{'AIML': np.float64(447.0), 'ECE': np.float64(429.6667), 'CSE': np.float64(142.0)}
AIML    447.0000
ECE     429.6667
CSE     142.0000
dtype: float64 

department ranks according to total marks:
AIML    1.0
ECE     2.0
CSE     3.0
dtype: float64 

{'AIML': np.float64(55.875), 'ECE': np.float64(53.708349999999996), 'CSE': np.float64(35.5)}
AIML    55.87500
ECE     53.70835
CSE     35.50000
dtype: float64 

department ranks according to average %:
AIML    1.0
ECE     2.0
CSE     3.0
dtype: float64


In [ ]:
# hardest subject
# subject with highest number of students scoring below it's average
# students_below_avg = {subject:number for subject,number in zip(list(dataset.iloc[:,:-7].columns), list((dataset.loc[:,dataset[sub]])<(dataset.loc[:,dataset[sub]].mean()).count() for sub in list(dataset.iloc[:,:-7].columns)))}
subject_cols = dataset.iloc[:,:-7].columns
# students_below_avg = {sub: (dataset[sub]<dataset[sub].mean()).sum() for sub in subject_cols}
students_below_avg = {sub: (dataset.index[dataset[sub]<dataset[sub].mean()],(dataset[sub]<dataset[sub].mean()).sum()) for sub in subject_cols}
print(students_below_avg)
subject_wise_number = {sub: (dataset[sub]<dataset[sub].mean()).sum() for sub in subject_cols}
print(subject_wise_number)
subjects_difficulty_wise = pd.Series(subject_wise_number)
print(subjects_difficulty_wise)
subjects_difficulty_ranks = subjects_difficulty_wise.rank(ascending=False, method="dense")
print(subjects_difficulty_ranks)
print("hardest subject with most students scoring below the average of the subject:", subjects_difficulty_ranks.index[0])

{'Python fundamentals': (Index(['Aaron', 'Ryan', 'Henry'], dtype='object', name='Students'), np.int64(3)), 'System Design': (Index(['Sam', 'Ryan'], dtype='object', name='Students'), np.int64(2)), 'Machine Learning': (Index(['Ryan', 'Henry'], dtype='object', name='Students'), np.int64(2)), 'Blockchain technology': (Index(['Sam', 'Henry'], dtype='object', name='Students'), np.int64(2))}
{'Python fundamentals': np.int64(3), 'System Design': np.int64(2), 'Machine Learning': np.int64(2), 'Blockchain technology': np.int64(2)}
Python fundamentals      3
System Design            2
Machine Learning         2
Blockchain technology    2
dtype: int64
Python fundamentals      1.0
System Design            2.0
Machine Learning         2.0
Blockchain technology    2.0
dtype: float64
hardest subject with most students scoring below the average of the subject: Python fundamentals
